### 1. Check if all data is downloaded correctly (Grand total: 13K rows)

In [1]:
import pandas as pd
from pathlib import Path

base_path = Path("../data/tabular_data/raw")

folders = ["cme", "wsj", "yahoo_finance"]

total_rows = 0

for folder in folders:
    folder_total = 0
    folder_path = base_path / folder
    
    for file in folder_path.glob("*.csv"):
        df = pd.read_csv(file)
        rows = len(df)
        folder_total += rows
        print(f"{file.name}: {rows}")
    
    print(f"{folder} total: {folder_total}\n")
    total_rows += folder_total

print("GRAND TOTAL:", total_rows)

Lean Hogs Future.csv: 7673
Chicago SRW Wheat Future.csv: 7644
KC HRW Wheat Future.csv: 6181
Soybean Oil Future.csv: 9998
Corn Future.csv: 9684
Live Cattle Future.csv: 6608
Class IV Milk Future.csv: 3617
Soybeans Future.csv: 10146
Class III Milk Future.csv: 9337
Feeder Cattle Future.csv: 5439
Soybean Meal Future.csv: 10295
cme total: 86622

us_30_year_bond_yield.csv: 1329
us_3_year_bond_yield.csv: 1403
us_2_year_bond_yield.csv: 1406
us_1_year_bond_yield.csv: 1402
us_7_year_bond_yield.csv: 1401
us_10_year_bond_yield.csv: 1400
us_5_year_bond_yield.csv: 1402
wsj total: 9743

Natural Gas.csv: 1218
S&P 500 Industrials.csv: 1215
S&P 500 Financials.csv: 1215
Russell 2000.csv: 1218
WTI Crude Oil.csv: 1218
Gold.csv: 1218
Euro versus Pound.csv: 1262
S&P 500 Communication Services.csv: 1215
S&P 500 Health Care.csv: 1215
British Pound.csv: 1261
Brent Crude Oil.csv: 1218
S&P 500 Energy.csv: 1218
Dow Jones Industrial Average.csv: 1218
MSCI World index.csv: 1215
Dollar Index.csv: 1218
S&P 500 Material

### 2. Visualise Embeddings for LLama 

In [ ]:
from embedding_atlas.widget import EmbeddingAtlasWidget
from datasets import load_dataset
import pandas as pd

data_path = "../data/tabular_data/raw/yahoo_finance/S&P 500.csv"
# Load your dataset from the Hub
df = pd.read_csv(data_path)

# Create interactive widget
widget = EmbeddingAtlasWidget(df)
widget

### 3. DEMO: DataShot style fact-extraction and fact-composition

#### 1. Load data for a specific day

In [ ]:
import pandas as pd
import os
from pathlib import Path
from datetime import datetime, timedelta

TARGET_DATE_STR = "2020-10-01"
BASE_PATH = Path("../data/tabular_data/report_table_data/injected/1week")
MAX_LOOKBACK_DAYS = 7 

# --- Dynamic Volatility Thresholds for different assest types ---
# 'perf' = Daily % change needed to trigger a Surge/Plunge
# 'vol' = Intraday range % needed to trigger High Volatility
THRESHOLDS = {
    'Treasury':  {'perf': 5.0,  'vol': 8.0},
    'Commodity': {'perf': 2.0,  'vol': 3.0},  # Oil/Agri are highly volatile
    'Equity':    {'perf': 1.25, 'vol': 2.0},
    'Currency':  {'perf': 0.4,  'vol': 0.6}   # Forex barely moves
}

def classify_asset(symbol):
    """Accurately maps the asset based on ticker, ignoring folder contamination."""
    symbol = str(symbol).upper()
    if 'TMUBMUSD' in symbol: return 'Treasury'
    if '=X' in symbol: return 'Currency'
    if symbol in ['CL=F', 'GC=F', 'NG=F', 'ZC', 'ZW', 'ZS', 'LE', 'HE', 'GF', 'DC', 'DK'] or len(symbol) <= 3: 
        return 'Commodity'
    return 'Equity' # S&P sectors, Nasdaq, individual stocks

def load_robust_data(target_date_str, base_path, max_lookback_days):
    target_date = datetime.strptime(target_date_str, "%Y-%m-%d")
    min_valid_date = target_date - timedelta(days=max_lookback_days)
    all_data = []
    
    for root, dirs, files in os.walk(base_path):
        if not files: continue
        valid_files = [f for f in files if f.endswith('.csv') and min_valid_date <= datetime.strptime(f.replace('.csv', ''), "%Y-%m-%d") <= target_date]
        
        if valid_files:
            valid_files.sort(reverse=True)
            df = pd.read_csv(os.path.join(root, valid_files[0]))
            all_data.append(df)
            
    if not all_data: return pd.DataFrame()
        
    master_df = pd.concat(all_data, ignore_index=True)
    df_latest = master_df.sort_values('Date').groupby('Symbol').last().reset_index()
    
    df_latest['Asset_Class'] = df_latest['Symbol'].apply(classify_asset)
    
    for ac, limits in THRESHOLDS.items():
        mask = df_latest['Asset_Class'] == ac
        
        # Recalculate Daily Performance
        df_latest.loc[mask, 'Daily_Performance'] = pd.cut(
            df_latest.loc[mask, 'pct_change'],
            bins=[-float('inf'), -limits['perf'], limits['perf'], float('inf')],
            labels=['Plunge', 'Flat', 'Surge']
        )
        
        # Recalculate Weekly Performance (Scaled slightly wider)
        df_latest.loc[mask, 'Weekly_Performance'] = pd.cut(
            df_latest.loc[mask, 'weekly_change'],
            bins=[-float('inf'), -limits['perf']*2.0, limits['perf']*2.0, float('inf')],
            labels=['Weekly Loss', 'Flat', 'Weekly Gain']
        )
        
        # Recalculate Volatility
        df_latest.loc[mask, 'Volatility_State'] = pd.cut(
            df_latest.loc[mask, 'intraday_range'],
            bins=[-float('inf'), limits['vol']/2, limits['vol'], float('inf')],
            labels=['Low Volatility', 'Normal', 'High Volatility']
        )
        
    return df_latest

df_latest = load_robust_data(TARGET_DATE_STR, BASE_PATH, MAX_LOOKBACK_DAYS)
print(f"Loaded {len(df_latest)} unique products. Asset Breakdown:")
print(df_latest['Asset_Class'].value_counts())

Loaded 35 unique products. Asset Breakdown:
Asset_Class
Equity       21
Treasury      7
Currency      5
Commodity     2
Name: count, dtype: int64


#### 2. Fact Extraction
- bottom-up statistical mining. We will group the data by Market (DataShot's "Subspace Enumeration") and calculate the distribution of our categorical variables (DataShot's "Proportion Facts")

In [28]:
def extract_enriched_facts(df, market_col, category_col, value_col):
    distribution = df.groupby([market_col, category_col], observed=False).size().unstack(fill_value=0)
    percentages = distribution.div(distribution.sum(axis=1), axis=0) * 100
    means = df.groupby([market_col, category_col], observed=False)[value_col].mean().unstack()
    impact = df.groupby(market_col).size()
    return percentages.round(1), means.round(2), impact

if not df_latest.empty:
    # Notice we swapped 'Market' for 'Asset_Class'
    perf_pct, perf_mean, impact = extract_enriched_facts(df_latest, 'Asset_Class', 'Daily_Performance', 'pct_change')
    trend_pct, trend_mean, _ = extract_enriched_facts(df_latest, 'Asset_Class', 'Trend_Status', 'dist_from_sma')
    vol_pct, vol_mean, _ = extract_enriched_facts(df_latest, 'Asset_Class', 'Volatility_State', 'intraday_range')
    weekly_pct, weekly_mean, _ = extract_enriched_facts(df_latest, 'Asset_Class', 'Weekly_Performance', 'weekly_change')
    rsi_pct, rsi_mean, _ = extract_enriched_facts(df_latest, 'Asset_Class', 'RSI_State', 'RSI')
    print("Aggregation successful.")

Aggregation successful.


#### 3. Fact Scoring and Filtering
- Scores facts to find anomalies. Define "Significant" as any category where the percentage exceeds 60% (a dominant trend) or is less than 5% (a rare outlier)

In [ ]:
SIGNIFICANCE_THRESHOLD_HIGH = 60.0 
MIN_IMPACT = 4 # Lower if variation of data classes is low

ALLOWED_MARKETS = {
    'Performance': ['Equity', 'Treasury', 'Commodity', 'Currency'],
    'Weekly': ['Equity', 'Treasury', 'Commodity', 'Currency'],
    'Trend': ['Equity', 'Commodity'], 
    'Volatility': ['Equity', 'Commodity'], 
    'RSI': []
}

def generate_scored_facts(df, pct_df, mean_df, impact_series, fact_type):
    scored_facts = []
    for asset_class, row in pct_df.iterrows():
        if impact_series.get(asset_class, 0) < MIN_IMPACT or asset_class not in ALLOWED_MARKETS[fact_type]:
            continue
            
        for category, percentage in row.items():
            if percentage >= SIGNIFICANCE_THRESHOLD_HIGH:
                avg_val = mean_df.loc[asset_class, category]
                text = ""
                
                if fact_type == 'Performance':
                    market_data = df[df['Asset_Class'] == asset_class].dropna(subset=['pct_change'])
                    if not market_data.empty:
                        best = market_data.loc[market_data['pct_change'].idxmax()]
                        worst = market_data.loc[market_data['pct_change'].idxmin()]
                        b_wk = best.get('weekly_change', 'N/A')
                        w_wk = worst.get('weekly_change', 'N/A')
                        
                        leader_str = f" The trend was led by {best['Product Name']} (+{best['pct_change']}% daily, {b_wk}% weekly), while {worst['Product Name']} lagged ({worst['pct_change']}% daily, {w_wk}% weekly)."
                    else:
                        leader_str = ""
                        
                    text = f"GROUP FACT: In {asset_class} assets, a highly significant {percentage}% of products exhibited a '{category}' in closing price, averaging a {avg_val}% daily change.{leader_str}"
                
                elif fact_type == 'Weekly':
                    text = f"GROUP FACT: Looking at the 5-day horizon for {asset_class} assets, {percentage}% recorded a '{category}', averaging a {avg_val}% weekly change."
                elif fact_type == 'Trend':
                    text = f"GROUP FACT: Across {asset_class} assets, {percentage}% are trending '{category}', sitting an average of {avg_val}% away from their 20-day moving average."
                elif fact_type == 'Volatility':
                    text = f"GROUP FACT: In {asset_class} assets, {percentage}% showed '{category}' intraday swings, with an average high-to-low range of {avg_val}%."
                elif fact_type == 'RSI':
                    text = f"GROUP FACT: Momentum in {asset_class} assets is skewed, with {percentage}% currently reading as '{category}' on the 14-day RSI (average RSI: {avg_val})."
                
                scored_facts.append((percentage, text))
    return scored_facts

group_facts = []
if not df_latest.empty:
    group_facts.extend(generate_scored_facts(df_latest, perf_pct, perf_mean, impact, 'Performance'))
    group_facts.extend(generate_scored_facts(df_latest, weekly_pct, weekly_mean, impact, 'Weekly'))
    group_facts.extend(generate_scored_facts(df_latest, trend_pct, trend_mean, impact, 'Trend'))
    group_facts.extend(generate_scored_facts(df_latest, vol_pct, vol_mean, impact, 'Volatility'))
    group_facts.extend(generate_scored_facts(df_latest, rsi_pct, rsi_mean, impact, 'RSI'))

#### 4. Entity level fact extraction


In [32]:
def extract_entity_facts(df):
    entity_texts = []
    benchmarks = {'^GSPC': 'S&P 500', '^IXIC': 'Nasdaq Composite'}
    
    # 1. Extract macro benchmark data
    for sym, name in benchmarks.items():
        row = df[df['Symbol'] == sym]
        if not row.empty:
            daily_chg = row.iloc[0]['pct_change']
            weekly_chg = row.iloc[0].get('weekly_change', 'N/A')
            entity_texts.append(f"MACRO ANCHOR: The benchmark {name} recorded a daily change of {daily_chg}% (Weekly change: {weekly_chg}%).")

    # 2. Extract global extreme outliers (With weekly change added)
    tradable_assets = df[~df['Symbol'].isin(benchmarks.keys())].copy()
    if len(tradable_assets) > 0:
        movers = tradable_assets.dropna(subset=['pct_change']).sort_values('pct_change', ascending=False)
        if not movers.empty:
            best = movers.iloc[0]
            worst = movers.iloc[-1]
            
            b_wk = best.get('weekly_change', 'N/A')
            w_wk = worst.get('weekly_change', 'N/A')
            
            entity_texts.append(f"GLOBAL OUTLIER: The top performing individual asset across all markets was {best['Product Name']} ({best['Symbol']}) with a {best['pct_change']}% daily change ({b_wk}% weekly).")
            entity_texts.append(f"GLOBAL OUTLIER: The worst performing individual asset was {worst['Product Name']} ({worst['Symbol']}) with a {worst['pct_change']}% daily change ({w_wk}% weekly).")
            
    return entity_texts

if not df_latest.empty:
    # Get mandatory entity facts
    final_entity_facts = extract_entity_facts(df_latest)
    
    # Sort group facts by significance (percentage) descending
    group_facts.sort(key=lambda x: x[0], reverse=True)
    
    # Calculate how many group facts we need to hit our max of 8
    slots_remaining = 8 - len(final_entity_facts)
    
    # Extract just the text from the top N scored group facts
    top_group_texts = [fact[1] for fact in group_facts[:slots_remaining]]
    
    # Final Payload Assembly
    final_rag_payload = final_entity_facts + top_group_texts
    
    print(f"--- FINAL RAG INJECTION PAYLOAD ({len(final_rag_payload)} Facts) ---")
    for idx, fact in enumerate(final_rag_payload, 1):
        print(f"{idx}. {fact}")

--- FINAL RAG INJECTION PAYLOAD (8 Facts) ---
1. MACRO ANCHOR: The benchmark S&P 500 recorded a daily change of 0.83% (Weekly change: 3.9%).
2. MACRO ANCHOR: The benchmark Nasdaq Composite recorded a daily change of 0.74% (Weekly change: 5.03%).
3. GLOBAL OUTLIER: The top performing individual asset across all markets was United States 3-Year Bond Yield (TMUBMUSD03Y) with a 18.88% daily change (4.94% weekly).
4. GLOBAL OUTLIER: The worst performing individual asset was WTI Crude Oil (CL=F) with a -3.73% daily change (-3.94% weekly).
5. GROUP FACT: Looking at the 5-day horizon for Treasury assets, 100.0% recorded a 'Flat', averaging a 2.99% weekly change.
6. GROUP FACT: In Equity assets, a highly significant 95.2% of products exhibited a 'Flat' in closing price, averaging a 0.52% daily change. The trend was led by S&P 500 Health Care (+1.68% daily, 3.58% weekly), while S&P 500 Industrials lagged (-0.26% daily, 2.23% weekly).
7. GROUP FACT: Across Equity assets, 90.5% are trending 'Testi